# Dehesa Tree Crown Segmentation (YOLOv11)

An automated deep learning-based tool for forest inventory and tree crown segmentation in Mediterranean "Dehesa" ecosystems. This project leverages **YOLOv11** and geo-spatial libraries to transform aerial imagery into actionable forestry data.

### Project Status: Beta & Preliminary Approach
This repository represents a **preliminary approximation** to automated crown detection. Please note:
* The current version is in **Beta**.
* Results and model precision are subject to ongoing refinement.
* The workflow is designed for research and testing purposes, not yet for production environments.

### Contributions & Feedback
I am actively looking to improve this tool! **Feedback, bug reports, and contributions are highly encouraged.** If you have suggestions regarding model optimization, geospatial logic, or edge-case handling in complex forest structures, please feel free to open an issue or submit a pull request.

---

## Key Features
* **Tiling & Overlap Logic**: Efficiently processes large high-resolution orthophotos by splitting them into manageable windows with vectorized NMS for duplicate suppression.
* **Geospatial Accuracy**: Uses Affine transformations to map pixel-level detections to real-world CRS coordinates.
* **Biometric Calculations**: Automatically computes:
    * Tree density (stems/ha).
    * Canopy Cover (CC %).
    * Individual and mean crown diameters (m).
* **GIS Integration**: Exports results directly to **GeoPackage (.gpkg)**, compatible with QGIS and ArcGIS.

---

## Installation

```bash
# 1. Install PyTorch matching your CUDA version (check with nvidia-smi)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

# 2. Install remaining dependencies
pip install -r requirements.txt
```

---

## Sample Inventory Statistics
The following results were generated from a sample analysis of a **75.22 hectare** study area:

| Metric | Result |
| :--- | :--- |
| **Detected Stems** | 3,643 trees |
| **Tree Density** | 48.43 stems/ha |
| **Canopy Cover (CC)** | 18.08 % |
| **Mean Crown Diameter** | 6.68 m |
| **Max Crown Diameter** | 13.44 m |

---

## Usage Example
The core logic is structured into modular classes for easy integration:

```python
from Crown_detector import CrownDetectionEngine, InventoryReporter

engine = CrownDetectionEngine(
    model_path     = "path/to/weights/best.pt",
    tile_size      = 960,
    conf_threshold = 0.25,
    min_diameter_m = 3.0
)

gdf_crowns, raster_meta = engine.detect("dehesa_ortho.tif")

InventoryReporter.compute_statistics(
    gdf         = gdf_crowns,
    raster_meta = raster_meta,
    output_path = "Forest_Inventory_Results.gpkg"
)
```


In [ ]:
import logging
import multiprocessing
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from threading import Lock
from typing import Dict, Iterator, List, Optional, Tuple
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from shapely.geometry import Polygon
from shapely.errors import TopologicalError
try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
except ImportError:
    CUDA_AVAILABLE = False
    warnings.warn("torch not found. Inference will run on CPU.", stacklevel=2)
from ultralytics import YOLO
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)



# TILE SAMPLER — generates windows without loading any image data
class TilingSampler:
    """
    Generates (col_off, row_off, width, height) tile descriptors for a raster
    using a sliding window with configurable overlap.

    No image data is read here — purely geometric window arithmetic.
    """

    def __init__(
        self,
        raster_width:  int,    # Full raster width in pixels
        raster_height: int,    # Full raster height in pixels
        tile_size:     int,    # Tile side length in pixels (square)
        overlap:       int     # Overlap between adjacent tiles in pixels
    ) -> None:
        if overlap >= tile_size:
            raise ValueError(
                f"overlap ({overlap}) must be less than tile_size ({tile_size})."
            )
        self.raster_width  = raster_width
        self.raster_height = raster_height
        self.tile_size     = tile_size
        self.stride        = tile_size - overlap  # effective step between tiles

    def __iter__(self) -> Iterator[Tuple[int, int, int, int]]:
        """Yields (col_off, row_off, tile_w, tile_h) for every valid tile."""
        for row_off in range(0, self.raster_height, self.stride):
            for col_off in range(0, self.raster_width, self.stride):
                tile_w = min(self.tile_size, self.raster_width  - col_off)
                tile_h = min(self.tile_size, self.raster_height - row_off)
                # Skip tiles too small for meaningful inference
                if tile_w >= 32 and tile_h >= 32:
                    yield col_off, row_off, tile_w, tile_h

    def __len__(self) -> int:
        """Total number of tiles (for progress reporting)."""
        cols = len(range(0, self.raster_width,  self.stride))
        rows = len(range(0, self.raster_height, self.stride))
        return cols * rows


# CROWN DETECTION ENGINE
class CrownDetectionEngine:
    """
    Parallel tile-based tree crown detector using a YOLOv11 segmentation model.

    Design:
        - Raster is read tile-by-tile via rasterio Window (out-of-core, bounded RAM).
        - Inference runs in a ThreadPoolExecutor: I/O-bound raster reads + GPU
          inference benefit from threads over processes (no GIL contention on GPU).
        - Detected masks are converted to georeferenced Polygon objects using the
          raster's affine transform (pixel → CRS coordinate).
        - Polygon batches are flushed to a GeoDataFrame every `flush_every` tiles
          to prevent unbounded list growth in RAM.
        - NMS is vectorized via spatial join + IoU threshold.

    Parameters
    ----------
    model_path      : Path to YOLOv11 .pt weights file.
    tile_size       : Inference tile size in pixels. Match model training size.
    overlap         : Tile overlap in pixels. Larger = fewer missed border crowns,
                      more duplicates to suppress. 200px is standard for 960px tiles.
    conf_threshold  : YOLO confidence threshold [0-1]. Lower = more detections,
                      more false positives.
    nms_iou_thresh  : Intersection-over-Union threshold for spatial NMS.
                      Detections with IoU > threshold are considered duplicates.
    min_diameter_m  : Minimum crown diameter in meters. Filters noise and scrub.
    n_workers       : Threads for parallel tile processing. -1 = CPU count.
    flush_every     : Tiles processed before flushing polygon buffer to GeoDataFrame.
    """

    def __init__(
        self,
        model_path:     str,
        tile_size:      int   = 960,
        overlap:        int   = 200,
        conf_threshold: float = 0.25,
        nms_iou_thresh: float = 0.50,
        min_diameter_m: float = 3.0,
        n_workers:      int   = -1,
        flush_every:    int   = 50
    ) -> None:
        self.model_path     = Path(model_path)
        self.tile_size      = tile_size
        self.overlap        = overlap
        self.conf_threshold = conf_threshold
        self.nms_iou_thresh = nms_iou_thresh
        self.min_diameter_m = min_diameter_m
        self.n_workers      = (
            max(1, multiprocessing.cpu_count()) if n_workers == -1 else max(1, n_workers)
        )
        self.flush_every    = flush_every

        self.device = "cuda:0" if CUDA_AVAILABLE else "cpu"
        logger.info(f"Inference device: {self.device}")

        logger.info(f"Loading YOLO model: {self.model_path.name}...")
        self.model = YOLO(str(self.model_path))

    def _read_tile(
        self,
        src:     rasterio.DatasetReader,
        col_off: int,
        row_off: int,
        tile_w:  int,
        tile_h:  int
    ) -> np.ndarray:
        """
        Reads RGB bands for a tile window. Thread-safe (rasterio read is GIL-safe).
        Returns HWC uint8 array for YOLO inference.
        """
        window = Window(col_off, row_off, tile_w, tile_h)
        # Read bands 1,2,3 (RGB) — moveaxis converts CHW → HWC
        return np.moveaxis(src.read([1, 2, 3], window=window), 0, -1)

    def _infer_tile(
        self,
        tile_img: np.ndarray
    ) -> Optional[List[np.ndarray]]:
        """
        Runs YOLO segmentation on a single tile image.
        Returns list of mask coordinate arrays (pixel space), or None.
        """
        results = self.model.predict(
            tile_img,
            imgsz   = self.tile_size,
            conf    = self.conf_threshold,
            device  = self.device,
            verbose = False
        )
        if results[0].masks is None:
            return None
        return [mask for mask in results[0].masks.xy if len(mask) >= 3]

    def _masks_to_polygons(
        self,
        masks:          List[np.ndarray],
        col_off:        int,
        row_off:        int,
        affine_transform
    ) -> List[Polygon]:
        """
        Converts tile-local pixel mask coordinates to georeferenced Polygons.

        Two-step coordinate transform:
            1. Tile-local pixel → global image pixel (add tile offset).
            2. Global image pixel → CRS coordinate (apply raster affine transform).
        """
        polygons: List[Polygon] = []
        for mask in masks:
            # Step 1: tile-local → global pixel
            global_pixels = [(p[0] + col_off, p[1] + row_off) for p in mask]
            # Step 2: global pixel → real-world CRS via affine transform
            geo_coords = [affine_transform * p for p in global_pixels]
            try:
                poly = Polygon(geo_coords)
                # buffer(0) repairs self-intersections without changing shape
                poly = poly.buffer(0)
                if poly.is_valid and not poly.is_empty:
                    polygons.append(poly)
            except (TopologicalError, ValueError):
                continue
        return polygons

    def _process_single_tile(
            self,
            image_path:      str,
            col_off:         int,
            row_off:         int,
            tile_w:          int,
            tile_h:          int,
            affine_transform
    ) -> List[Polygon]:
        """
        Full pipeline for one tile: read → infer → convert to polygons.
        Each thread opens its own rasterio handle to avoid JPEG thread-safety issues.
        """
        try:
            with rasterio.open(image_path) as src:
                tile_img = self._read_tile(src, col_off, row_off, tile_w, tile_h)
            masks = self._infer_tile(tile_img)
            if masks is None:
                return []
            return self._masks_to_polygons(masks, col_off, row_off, affine_transform)
        except Exception as e:
            logger.debug(f"Tile ({col_off},{row_off}) failed: {e}")
            return []

    def _vectorized_nms(self, gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
        """
        Removes duplicate detections from overlapping tiles using vectorized NMS.
        Much faster than iterrows() on large detection sets (>1000 polygons).
        """
        if len(gdf) == 0:
            return gdf

        logger.info(f"Running vectorized NMS on {len(gdf):,} raw detections...")

        # Sort descending by area — larger detections win ties
        gdf = gdf.copy()
        gdf["_area"] = gdf.geometry.area
        gdf = gdf.sort_values("_area", ascending=False).reset_index(drop=True)
        gdf["_id"] = gdf.index

        # Spatial join: find all pairs whose bounding boxes overlap
        joined = gpd.sjoin(
            gdf[["_id", "_area", "geometry"]],
            gdf[["_id", "_area", "geometry"]],
            how="inner",
            predicate="intersects"
        )

        # Keep only pairs where right detection is smaller (avoids symmetric duplicates)
        pairs = joined[joined["_id_left"] < joined["_id_right"]].copy()

        if len(pairs) == 0:
            gdf = gdf.drop(columns=["_area", "_id"])
            return gdf

        # Compute IoU for each candidate pair
        def compute_iou(row) -> float:
            try:
                geom_l = gdf.loc[row["_id_left"],  "geometry"]
                geom_r = gdf.loc[row["_id_right"], "geometry"]
                inter  = geom_l.intersection(geom_r).area
                union  = geom_l.union(geom_r).area
                return inter / union if union > 0 else 0.0
            except (TopologicalError, ValueError):
                return 0.0

        pairs["_iou"] = pairs.apply(compute_iou, axis=1)

        # IDs to suppress: smaller detection in any pair exceeding IoU threshold
        suppress_ids = set(
            pairs.loc[pairs["_iou"] > self.nms_iou_thresh, "_id_right"].tolist()
        )

        gdf_clean = gdf[~gdf["_id"].isin(suppress_ids)].drop(
            columns=["_area", "_id"]
        ).reset_index(drop=True)

        logger.info(
            f"NMS complete: {len(gdf):,} raw → {len(gdf_clean):,} unique detections "
            f"(removed {len(gdf) - len(gdf_clean):,} duplicates)"
        )
        return gdf_clean

    def detect(
        self,
        image_path: str
    ) -> Tuple[gpd.GeoDataFrame, Dict]:
        """
        Runs the full detection pipeline on a large orthophoto.

        Pipeline:
            1. Open raster (out-of-core — no full image load).
            2. Generate tile windows via TilingSampler.
            3. Process tiles in parallel threads.
               Batches of `flush_every` tiles flushed to GDF.
            4. Global vectorized NMS.
            5. Diameter filter and attribute computation.

        Parameters
        ----------
        image_path : Path to the input GeoTIFF orthophoto.

        Returns
        -------
        (GeoDataFrame of detected crowns, dict of raster metadata)
        """
        path = Path(image_path)
        if not path.exists():
            raise FileNotFoundError(f"Image not found: {path}")

        logger.info(f"Opening raster: {path.name}")

        with rasterio.open(str(path)) as src:
            affine  = src.transform
            crs     = src.crs
            width   = src.width
            height  = src.height
            res_x, res_y = src.res

            # FIX-6: compute valid-pixel area from data mask, not bounding box
            # Read subsampled mask to avoid loading full raster
            scale       = 10
            data_mask   = src.dataset_mask(
                out_shape=(height // scale + 1, width // scale + 1)
            )
            valid_frac  = (data_mask > 0).mean()
            total_area_ha = (width * height * abs(res_x * res_y) * valid_frac) / 10000

            raster_meta = {
                "width":          width,
                "height":         height,
                "res_m":          abs(res_x),
                "crs":            crs,
                "total_area_ha":  total_area_ha
            }

            logger.info(
                f"Raster: {width}x{height} px | GSD={abs(res_x)*100:.2f} cm/px | "
                f"Valid area={total_area_ha:.4f} ha"
            )

            sampler   = TilingSampler(width, height, self.tile_size, self.overlap)
            all_tiles = list(sampler)
            n_tiles   = len(all_tiles)
            logger.info(
                f"Tiling: {n_tiles} tiles | "
                f"tile_size={self.tile_size} | overlap={self.overlap} | "
                f"workers={self.n_workers}"
            )

            # Threads are optimal here: rasterio reads release GIL,
            # GPU inference is non-blocking from Python's perspective.
            polygon_batches: List[gpd.GeoDataFrame] = []
            current_batch:   List[Polygon]          = []
            batch_lock = Lock()
            tiles_done = 0

            def process_tile_task(tile_args: Tuple) -> List[Polygon]:
                col_off, row_off, tile_w, tile_h = tile_args
                return self._process_single_tile(
                    image_path,    # la variable path del detect(), no src
                    col_off, row_off, tile_w, tile_h, affine
    )

            with ThreadPoolExecutor(max_workers=self.n_workers) as executor:
                future_to_tile = {
                    executor.submit(process_tile_task, tile): tile
                    for tile in all_tiles
                }

                for future in as_completed(future_to_tile):
                    polys = future.result()
                    tiles_done += 1

                    with batch_lock:
                        current_batch.extend(polys)

                        if len(current_batch) > 0 and tiles_done % self.flush_every == 0:
                            polygon_batches.append(
                                gpd.GeoDataFrame(
                                    {"geometry": current_batch}, crs=crs
                                )
                            )
                            current_batch = []

                    if tiles_done % 100 == 0 or tiles_done == n_tiles:
                        logger.info(
                            f"  Tiles: {tiles_done}/{n_tiles} | "
                            f"polygons in buffer: {len(current_batch):,}"
                        )

                # Flush remaining polygons
                if current_batch:
                    polygon_batches.append(
                        gpd.GeoDataFrame({"geometry": current_batch}, crs=crs)
                    )

        if not polygon_batches:
            logger.warning("No tree crowns detected.")
            return gpd.GeoDataFrame({"geometry": []}, crs=crs), raster_meta

        # Consolidate all batches into one GeoDataFrame
        gdf_raw = pd.concat(polygon_batches, ignore_index=True)
        gdf_raw = gpd.GeoDataFrame(gdf_raw, geometry="geometry", crs=crs)
        logger.info(f"Total raw detections: {len(gdf_raw):,}")

        # Global vectorized NMS (FIX-2)
        gdf_clean = self._vectorized_nms(gdf_raw)

        # Diameter filter
        gdf_clean["area_sqm"]      = gdf_clean.geometry.area.round(2)
        gdf_clean["crown_diam_m"]  = (
            2.0 * np.sqrt(gdf_clean["area_sqm"] / np.pi)
        ).round(2)
        gdf_clean = gdf_clean[
            gdf_clean["crown_diam_m"] >= self.min_diameter_m
        ].copy().reset_index(drop=True)

        logger.info(
            f"After diameter filter (>= {self.min_diameter_m}m): "
            f"{len(gdf_clean):,} crowns retained"
        )

        return gdf_clean, raster_meta


# INVENTORY REPORTER
class InventoryReporter:
    """
    Computes dasometric statistics and exports results to GeoPackage.

    Separating reporting from detection keeps CrownDetectionEngine focused
    and allows the reporter to be swapped for different output formats.
    """

    @staticmethod
    def compute_statistics(
        gdf:           gpd.GeoDataFrame,
        raster_meta:   Dict,
        output_path:   str
    ) -> gpd.GeoDataFrame:
        """
        Adds ID and centroid coordinates, computes forest statistics,
        and exports the final GeoPackage.

        Parameters
        ----------
        gdf          : Cleaned crown GeoDataFrame from CrownDetectionEngine.
        raster_meta  : Dict with raster metadata (area_ha, res_m, crs, etc.).
        output_path  : Path for the output .gpkg file.

        Returns
        -------
        Enriched GeoDataFrame with ID and coordinate columns.
        """
        if len(gdf) == 0:
            logger.warning("Empty GeoDataFrame — nothing to report.")
            return gdf

        gdf = gdf.copy().reset_index(drop=True)
        gdf["id"]          = range(1, len(gdf) + 1)
        centroids          = gdf.geometry.centroid
        gdf["coord_x"]     = centroids.x.round(2)
        gdf["coord_y"]     = centroids.y.round(2)

        total_area_ha = raster_meta["total_area_ha"]
        diameters     = gdf["crown_diam_m"]
        canopy_cover  = (gdf["area_sqm"].sum() / (total_area_ha * 10_000)) * 100
        density       = len(gdf) / total_area_ha

        print("\n" + "=" * 50)
        print("       FOREST INVENTORY STATISTICS")
        print("=" * 50)
        print(f"  GSD                  : {raster_meta['res_m']*100:.2f} cm/px")
        print(f"  Valid area analyzed  : {total_area_ha:.4f} ha")
        print(f"  Detected stems       : {len(gdf):,} trees")
        print(f"  Density              : {density:.2f} stems/ha")
        print(f"  Canopy Cover (FCC)   : {canopy_cover:.2f} %")
        print("-" * 50)
        print(f"  Mean crown diameter  : {diameters.mean():.2f} m")
        print(f"  Std deviation (diam) : {diameters.std():.2f} m")
        print(f"  Min diameter         : {diameters.min():.2f} m")
        print(f"  Max diameter         : {diameters.max():.2f} m")
        print(f"  Median diameter      : {diameters.median():.2f} m")
        print("=" * 50)

        out_path = Path(output_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)

        gdf[["id", "area_sqm", "crown_diam_m", "coord_x", "coord_y", "geometry"]].to_file(
            str(out_path), driver="GPKG"
        )
        logger.info(f"GeoPackage exported: {out_path.name}")

        return gdf


# ENTRY POINT
if __name__ == "__main__":

    # ------------------------------------------------------------------
    # Configuration — all parameters documented, no hardcoded paths
    # ------------------------------------------------------------------
    HERE        = Path(r"C:/path/to/your/project")            # ← set your project directory
    IMAGE_PATH  = str(HERE / "orthophoto.tif")                # Input GeoTIFF
    MODEL_PATH  = str(HERE / "models/Nano_3_960/weights/best.pt")  # YOLOv11 weights
    OUTPUT_GPKG = str(HERE / "Forest_Inventory_Results.gpkg") # Output GeoPackage

    engine = CrownDetectionEngine(
        model_path     = MODEL_PATH,
        tile_size      = 960,     # Match YOLO training imgsz
        overlap        = 200,     # Overlap in pixels — larger reduces missed borders
        conf_threshold = 0.2,    # YOLO confidence [0-1] —> lower = more detections
        nms_iou_thresh = 0.50,    # IoU threshold for duplicate suppression
        min_diameter_m = 3.0,     # Minimum crown diameter filter in meters
        n_workers      = 10,      # -1 = all CPU threads -> 10 if your kernel die (OOM)
        flush_every    = 20       # Flush polygon buffer every N tiles (RAM control)
    )

    gdf_crowns, meta = engine.detect(IMAGE_PATH)

    InventoryReporter.compute_statistics(
        gdf        = gdf_crowns,
        raster_meta = meta,
        output_path = OUTPUT_GPKG
    )
